# TMC-LM Training on Colab (T4 GPU)
Run all cells: `Runtime > Run all` | Need GPU: `Runtime > Change runtime type > T4 GPU`

Repo: `McEmil1993/tmc-llm` | Branch `main` | Dataset 410 examples (train.txt + Student-Manual_revised.pdf)

In [ ]:
# 1. Clone repo (kung wala pa)
!git clone https://github.com/McEmil1993/tmc-llm.git
%cd tmc-llm
!git pull origin main
!ls -lh data/raw/tmc_sources/

In [ ]:
# 2. Check GPU (dapat T4)
!nvidia-smi
import torch; print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pip show transformers peft accelerate | grep -E "Name|Version"

In [ ]:
# 4. Build dataset (410 examples = 286 train / 62 val / 62 test)
!python -m tmc_llm.dataset_builder --source-dir data/raw/tmc_sources --output-dir data/processed
!cat data/processed/metadata.json
!cat data/processed/sources_manifest.json
!wc -l data/processed/train.jsonl data/processed/validation.jsonl

In [ ]:
# 5. Train LoRA - pinaka dugay ~15-20 mins sa T4 GPU (80 steps, configs/train_lora.yaml)
!python -m tmc_llm.train_lora --config configs/train_lora.yaml
!ls -lh models/adapters/tmc-lm-tinyllama-lora/

In [ ]:
# 6. Merge LoRA -> HF merged model
!python -m tmc_llm.merge_lora --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --adapter-dir models/adapters/tmc-lm-tinyllama-lora --output-dir models/merged/tmc-lm-tinyllama
!ls -lh models/merged/tmc-lm-tinyllama/ | head -20

In [ ]:
# 7. Convert to GGUF (need llama.cpp - clone if wala)
!test -d external/llama.cpp || git clone https://github.com/ggml-org/llama.cpp external/llama.cpp
!cmake -B external/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -S external/llama.cpp && cmake --build external/llama.cpp/build --config Release -j$(nproc)
!python external/llama.cpp/convert_hf_to_gguf.py models/merged/tmc-lm-tinyllama --outfile models/gguf/tmc-lm-tinyllama-f16.gguf --outtype f16
!external/llama.cpp/build/bin/llama-quantize models/gguf/tmc-lm-tinyllama-f16.gguf models/gguf/tmc-lm-tinyllama-q4_k_m.gguf Q4_K_M
!python -m tmc_llm.gguf_check --path models/gguf/tmc-lm-tinyllama-q4_k_m.gguf
!ls -lh models/gguf/

In [ ]:
# 8. Save to Google Drive (para dili mawala)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/tmc-llm-models"
!cp -r models/adapters/tmc-lm-tinyllama-lora "/content/drive/MyDrive/tmc-llm-models/"
!cp -r models/merged/tmc-lm-tinyllama "/content/drive/MyDrive/tmc-llm-models/"
!cp -r models/gguf "/content/drive/MyDrive/tmc-llm-models/"
!cp -r data/processed "/content/drive/MyDrive/tmc-llm-models/"
!ls -lh "/content/drive/MyDrive/tmc-llm-models/"

### Next: Test sa Colab
`!python -c "from transformers import AutoModelForCausalLM; print('ok')"`

### Pull back to VSCode
Sa VSCode: `git pull origin main` kung naa kay gi-update sa Colab (kung gi-push).